# Flight Delay Analysis - Phase 2: Feature Engineering & Exploratory Data Analysis

## Consolidated EDA Notebook

**Project**: University Case Study - Flight Delay Analysis  
**Phase**: Phase 2 - Feature Engineering & Visual Analysis  
**Approach**: Consolidated notebook combining feature engineering and EDA  
**Input**: `data/processed/flights_active.parquet` (from Phase 1)

---

## Design Decision

**Consolidated Approach Rationale**: Feature engineering and EDA are combined in ONE notebook to:
- Reduce file management overhead
- Ensure analytical variables are created immediately before visualization
- Maintain clear lineage from raw features to insights
- Simplify reproducibility

---

## Table of Contents
1. [Setup & Data Loading](#part1)
2. [Feature Engineering](#part2)
3. [Exploratory Data Analysis](#part3)
4. [Insights & Findings](#part4)

---
<a id='part1'></a>
# PART 1: SETUP & DATA LOADING

## Section 1.1: Environment Setup

Import all required libraries and configure visualization settings.

In [ ]:
# Core data processing libraries
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Configure visualization styling

# Matplotlib/Seaborn settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# Plotly settings
import plotly.io as pio
pio.templates.default = "plotly_white"

# Pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print("✓ Visualization settings configured")

In [ ]:
# Define project paths
PROJECT_ROOT = Path('/home/user/Flight-delay')
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'

# Create reports directory if it doesn't exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths:")
print(f"  Processed data: {DATA_PROCESSED}")
print(f"  Reports output: {REPORTS_DIR}")

## Section 1.2: Load Processed Data

Load the cleaned active flights data from Phase 1.

In [ ]:
# Load active flights data
flights_file = DATA_PROCESSED / 'flights_active.parquet'

if not flights_file.exists():
    print("⚠️  ERROR: flights_active.parquet not found!")
    print("Please run Phase 1 notebook (data_ingestion_cleaning.ipynb) first.")
    raise FileNotFoundError(f"Required file not found: {flights_file}")

print(f"Loading data from: {flights_file}")
df = pd.read_parquet(flights_file)

print(f"\n✓ Data loaded successfully!")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Display basic information
print("=== DATASET INFO ===")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Display first 5 rows
print("=== FIRST 5 ROWS ===")
display(df.head())

In [ ]:
# Display last 5 rows to verify data integrity
print("=== LAST 5 ROWS ===")
display(df.tail())

In [ ]:
# Check data types
print("=== DATA TYPES ===")
print(df.dtypes)

In [ ]:
# Check for required columns
required_cols = ['DEP_TIME', 'MONTH', 'DISTANCE', 'ORIGIN', 'DEST', 'DEP_DELAY', 'ARR_DELAY', 'OP_CARRIER']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"⚠️  Warning: Expected columns not found: {missing_cols}")
    print("\nAvailable columns:")
    print(df.columns.tolist())
    print("\nNote: Column names may vary. Adjust feature engineering code accordingly.")
else:
    print("✓ All required columns present")

# Also check for alternative column names
if 'OP_CARRIER' not in df.columns and 'CARRIER' in df.columns:
    df['OP_CARRIER'] = df['CARRIER']
    print("✓ Mapped CARRIER to OP_CARRIER")

### ✅ Data Load Confirmation

Data has been successfully loaded from Phase 1 output. The dataset contains active (non-cancelled) flights ready for feature engineering and analysis.

---
<a id='part2'></a>
# PART 2: IN-MEMORY FEATURE ENGINEERING

## Feature Engineering for Analysis

**Rationale**: All analytical variables are created here before visualization to ensure consistency and reproducibility. Features are engineered in-memory to support subsequent visual analysis without requiring persistent storage.

**Strategy**:
1. Create temporal binning features (time blocks, seasons)
2. Create route complexity features (haul types, route IDs, hub flags)
3. Create operational stress metrics (volume aggregations)
4. Validate feature quality and document results

**Total Features to Create**: 7

In [ ]:
# Record initial memory usage
initial_memory_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Initial dataframe memory usage: {initial_memory_mb:.2f} MB")
print(f"Initial shape: {df.shape}")

## Section 2.1: Temporal Binning Features

### Feature 1: Time_Block

**Purpose**: Identify if delays correlate with daily rush periods (morning commute, evening rush, overnight operations).

**Hypothesis**: Delays may be concentrated during high-traffic periods when airports and airspace are most congested.

**Categories**:
- **Morning** (05:00 – 11:59): Early departures and business travel
- **Afternoon** (12:00 – 16:59): Mid-day operations
- **Evening** (17:00 – 21:59): Evening rush and return travel
- **Night** (22:00 – 04:59): Red-eye and overnight flights

In [ ]:
# Create Time_Block feature

def categorize_time_block(dep_time):
    """
    Categorize departure time into time blocks.
    
    Args:
        dep_time: Departure time (format may vary: HHMM as int, or datetime)
    
    Returns:
        str: Time block category
    """
    # Handle missing values
    if pd.isna(dep_time):
        return 'Unknown'
    
    # Convert to hour (handle both integer HHMM format and datetime)
    if isinstance(dep_time, (int, float)):
        hour = int(dep_time) // 100  # HHMM format: 1430 -> 14
    else:
        hour = dep_time.hour  # datetime format
    
    # Categorize by hour
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 22:
        return 'Evening'
    else:  # 22-04
        return 'Night'

# Apply function
df['Time_Block'] = df['DEP_TIME'].apply(categorize_time_block)

# Verify results
print("✓ Time_Block feature created")
print("\nDistribution:")
print(df['Time_Block'].value_counts().sort_index())
print(f"\nMissing values: {df['Time_Block'].isna().sum()}")

### Feature 2: Season

**Purpose**: Detect seasonal delay patterns related to weather, holidays, and travel volume.

**Hypothesis**: Winter may have more weather-related delays; summer may have higher volume-related delays.

**Categories**:
- **Winter** (Dec, Jan, Feb): Cold weather, holiday travel
- **Spring** (Mar, Apr, May): Moderate weather, spring break
- **Summer** (Jun, Jul, Aug): Peak vacation travel
- **Fall** (Sep, Oct, Nov): Back-to-school, Thanksgiving

In [ ]:
# Create Season feature

def categorize_season(month):
    """
    Categorize month into meteorological season.
    
    Args:
        month: Month number (1-12)
    
    Returns:
        str: Season category
    """
    if pd.isna(month):
        return 'Unknown'
    
    month = int(month)
    
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    elif month in [9, 10, 11]:
        return 'Fall'
    else:
        return 'Unknown'

# Apply function
df['Season'] = df['MONTH'].apply(categorize_season)

# Verify results
print("✓ Season feature created")
print("\nDistribution:")
print(df['Season'].value_counts().sort_index())
print(f"\nMissing values: {df['Season'].isna().sum()}")

### Temporal Features: Justification

**Why these temporal bins are meaningful**:

1. **Time_Block**: Aviation operations follow predictable daily patterns:
   - Morning: High business travel demand, fresh crews, minimal cascade delays
   - Afternoon: Cumulative delays from morning, crew fatigue
   - Evening: Peak congestion, compounded delays, weather buildups
   - Night: Reduced traffic, maintenance windows, red-eye operations

2. **Season**: Multiple seasonal factors affect delays:
   - Winter: Severe weather (snow, ice), de-icing delays
   - Spring: Thunderstorms, spring break volume spikes
   - Summer: Peak vacation demand, afternoon heat delays
   - Fall: Moderate weather, Thanksgiving travel surge

These categories balance granularity with statistical power - too many categories would fragment the data, too few would mask patterns.

## Section 2.2: Route Complexity Features

### Feature 3: Haul_Type

**Purpose**: Test if longer flights have different delay recovery patterns.

**Hypothesis**: Longer flights may have more opportunities to make up time in the air, while short flights are more sensitive to ground delays.

**Distance Thresholds** (based on industry standards and FAA definitions):
- **Short-Haul** (< 800 miles): Regional flights, limited cruise time
- **Medium-Haul** (800 – 2,200 miles): Transcontinental segments, moderate cruise
- **Long-Haul** (> 2,200 miles): Coast-to-coast and beyond, extended cruise

**Justification**: These thresholds align with:
- FAA distance-based fee structures
- Aircraft range categories
- Operational planning standards

In [ ]:
# Create Haul_Type feature

def categorize_haul_type(distance):
    """
    Categorize flight distance into haul types.
    
    Args:
        distance: Flight distance in miles
    
    Returns:
        str: Haul type category
    """
    if pd.isna(distance):
        return 'Unknown'
    
    if distance < 800:
        return 'Short-Haul'
    elif distance < 2200:
        return 'Medium-Haul'
    else:
        return 'Long-Haul'

# Apply function
df['Haul_Type'] = df['DISTANCE'].apply(categorize_haul_type)

# Verify results
print("✓ Haul_Type feature created")
print("\nDistribution:")
print(df['Haul_Type'].value_counts())
print(f"\nMissing values: {df['Haul_Type'].isna().sum()}")
print("\nDistance ranges by haul type:")
print(df.groupby('Haul_Type')['DISTANCE'].agg(['min', 'max', 'mean', 'count']))

### Feature 4: Route_ID

**Purpose**: Identify specific routes for bottleneck analysis.

**Format**: "ORIGIN-DEST" (e.g., "JFK-LAX")

**Use Cases**:
- Identify consistently problematic routes
- Analyze route-specific delay patterns
- Compare directional effects (JFK-LAX vs LAX-JFK)

In [ ]:
# Create Route_ID feature

df['Route_ID'] = df['ORIGIN'].astype(str) + '-' + df['DEST'].astype(str)

# Verify results
print("✓ Route_ID feature created")
print(f"\nTotal unique routes: {df['Route_ID'].nunique():,}")
print("\nTop 10 most frequent routes:")
print(df['Route_ID'].value_counts().head(10))

### Feature 5: Is_Hub

**Purpose**: Isolate hub congestion effects on delays.

**Hypothesis**: Major hub airports may experience more delays due to:
- Higher traffic volume
- More complex operations
- Greater exposure to cascade delays

**Methodology**: Dynamic calculation (not hardcoded):
1. Calculate total departure volume for each airport
2. Identify top 20 airports by volume
3. Flag these as hubs (True), all others as non-hubs (False)

In [ ]:
# Create Is_Hub feature (dynamic calculation)

# Calculate departure volume by airport
airport_volumes = df['ORIGIN'].value_counts()

# Identify top 20 airports
top_20_airports = set(airport_volumes.head(20).index)

print("Top 20 hub airports identified:")
print(airport_volumes.head(20))

# Create hub flag
df['Is_Hub'] = df['ORIGIN'].isin(top_20_airports)

# Verify results
print("\n✓ Is_Hub feature created")
print(f"\nHub flights: {df['Is_Hub'].sum():,} ({df['Is_Hub'].sum()/len(df)*100:.1f}%)")
print(f"Non-hub flights: {(~df['Is_Hub']).sum():,} ({(~df['Is_Hub']).sum()/len(df)*100:.1f}%)")

## Section 2.3: Operational Stress Metrics

**Purpose**: Create aggregated volume metrics to test the congestion hypothesis.

**Congestion Hypothesis**: Delays increase when operational capacity is stressed by high flight volumes.

**Why Daily Aggregates (not rolling windows)**:
- Memory constraints: Rolling windows require additional storage and computation
- Daily patterns: Airport and airline operations are planned on daily schedules
- Sufficient granularity: Daily aggregates capture volume stress without excessive detail
- Alignment with operational planning: Airlines and airports manage capacity daily

### Feature 6: Daily_Airport_Departures

**Purpose**: Measure daily operational load at each airport.

**Hypothesis**: Airports operating near capacity will have higher delay rates.

**Calculation**: For each (ORIGIN, DATE) pair, count total departures.

In [ ]:
# Create Daily_Airport_Departures feature

# First, ensure we have a date column
# Try to find date-related columns
date_cols = [col for col in df.columns if 'DATE' in col.upper() or 'FL_DATE' in col.upper()]

if date_cols:
    date_col = date_cols[0]
    print(f"Using date column: {date_col}")
else:
    # Try to construct from year, month, day if available
    if all(col in df.columns for col in ['YEAR', 'MONTH', 'DAY_OF_MONTH']):
        df['FL_DATE'] = pd.to_datetime(df[['YEAR', 'MONTH', 'DAY_OF_MONTH']].rename(
            columns={'YEAR': 'year', 'MONTH': 'month', 'DAY_OF_MONTH': 'day'}))
        date_col = 'FL_DATE'
        print("✓ Created FL_DATE from YEAR, MONTH, DAY_OF_MONTH")
    else:
        print("⚠️  Warning: No date column found. Using MONTH as proxy.")
        date_col = 'MONTH'

# Calculate daily airport departures
daily_airport_load = df.groupby(['ORIGIN', date_col]).size().reset_index(name='Daily_Airport_Departures')

# Merge back to main dataframe
df = df.merge(daily_airport_load, on=['ORIGIN', date_col], how='left')

# Verify results
print("\n✓ Daily_Airport_Departures feature created")
print("\nStatistics:")
print(df['Daily_Airport_Departures'].describe())
print(f"\nMissing values: {df['Daily_Airport_Departures'].isna().sum()}")

### Feature 7: Daily_Carrier_Load

**Purpose**: Measure daily operational load for each airline.

**Hypothesis**: Airlines operating near fleet capacity will have higher delay rates.

**Calculation**: For each (CARRIER, DATE) pair, count total flights.

In [ ]:
# Create Daily_Carrier_Load feature

# Calculate daily carrier load
daily_carrier_load = df.groupby(['OP_CARRIER', date_col]).size().reset_index(name='Daily_Carrier_Load')

# Merge back to main dataframe
df = df.merge(daily_carrier_load, on=['OP_CARRIER', date_col], how='left')

# Verify results
print("✓ Daily_Carrier_Load feature created")
print("\nStatistics:")
print(df['Daily_Carrier_Load'].describe())
print(f"\nMissing values: {df['Daily_Carrier_Load'].isna().sum()}")

## Section 2.4: Feature Engineering Summary & Memory Cleanup

In [ ]:
# Memory management: Delete intermediate dataframes
del daily_airport_load, daily_carrier_load, airport_volumes

import gc
gc.collect()

print("✓ Intermediate variables cleaned up")

In [ ]:
# Calculate final memory usage
final_memory_mb = df.memory_usage(deep=True).sum() / 1024**2
memory_increase = final_memory_mb - initial_memory_mb

print("=" * 70)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 70)

print("\n📊 NEW FEATURES CREATED (7 total):")
new_features = ['Time_Block', 'Season', 'Haul_Type', 'Route_ID', 'Is_Hub', 
                'Daily_Airport_Departures', 'Daily_Carrier_Load']

for i, feature in enumerate(new_features, 1):
    if feature in df.columns:
        null_pct = df[feature].isna().sum() / len(df) * 100
        unique_count = df[feature].nunique()
        print(f"  {i}. {feature}")
        print(f"     - Unique values: {unique_count:,}")
        print(f"     - Missing: {null_pct:.2f}%")

print(f"\n💾 MEMORY USAGE:")
print(f"  Initial: {initial_memory_mb:.2f} MB")
print(f"  Final: {final_memory_mb:.2f} MB")
print(f"  Increase: {memory_increase:.2f} MB ({memory_increase/initial_memory_mb*100:.1f}%)")

print(f"\n📏 DATAFRAME SHAPE:")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)} (added {len(new_features)} new features)")

print("\n✅ READY FOR VISUALIZATION PHASE")
print("=" * 70)

---
<a id='part3'></a>
# PART 3: EXPLORATORY DATA ANALYSIS (VISUALIZATIONS)

## Visual Analysis: 8-10 Complex Visualizations

**Visualization Principles**:
- **Effectiveness**: Choose chart types that best represent the data relationships
- **Expressiveness**: Ensure visualizations accurately convey the underlying patterns
- **Clarity**: Use clear labels, appropriate colors, and helpful annotations

**Quality Standards**:
- Descriptive titles explaining the insight
- Labeled axes with units
- Colorblind-friendly palettes
- Appropriate figure sizes for readability
- Statistical annotations where relevant

All visualizations will be saved to `reports/` directory for inclusion in final report.

In [ ]:
# Helper function to save visualizations
def save_viz(fig, filename, dpi=300):
    """
    Save visualization to reports directory.
    
    Args:
        fig: Matplotlib or Plotly figure object
        filename: Output filename (without extension)
        dpi: Resolution for saved image
    """
    filepath = REPORTS_DIR / f"{filename}.png"
    
    # Handle both matplotlib and plotly figures
    if hasattr(fig, 'write_image'):  # Plotly figure
        fig.write_image(str(filepath), width=1200, height=600)
    else:  # Matplotlib figure
        fig.savefig(filepath, dpi=dpi, bbox_inches='tight')
    
    print(f"✓ Saved: {filepath.name}")

print("Helper function loaded")

## Category 1: Univariate Analysis

Examine single variables to understand distributions and central tendencies.

### Visualization 1: Distribution of Departure Delays

**Purpose**: Understand the shape of delay distribution and identify skewness, outliers, and central tendency.

**Chart Type**: Histogram with overlaid KDE (Kernel Density Estimation)

In [ ]:
# Visualization 1: Departure Delay Distribution

fig, ax = plt.subplots(figsize=(12, 6))

# Filter extreme outliers for better visualization (keep 99th percentile)
delay_99th = df['DEP_DELAY'].quantile(0.99)
df_plot = df[df['DEP_DELAY'] <= delay_99th]

# Create histogram with KDE
sns.histplot(data=df_plot, x='DEP_DELAY', bins=100, kde=True, ax=ax, color='steelblue')

# Add vertical line at 15-minute threshold
ax.axvline(x=15, color='red', linestyle='--', linewidth=2, label='Delay Threshold (15 min)')

# Add mean and median lines
mean_delay = df['DEP_DELAY'].mean()
median_delay = df['DEP_DELAY'].median()
ax.axvline(x=mean_delay, color='orange', linestyle='--', linewidth=1.5, label=f'Mean ({mean_delay:.1f} min)')
ax.axvline(x=median_delay, color='green', linestyle='--', linewidth=1.5, label=f'Median ({median_delay:.1f} min)')

# Labels and title
ax.set_xlabel('Departure Delay (minutes)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Flight Departure Delays\nShowing Right-Skewed Pattern with Long Tail', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_viz(fig, 'viz_01_delay_distribution')
plt.show()

# Statistics
print("\n=== DELAY DISTRIBUTION STATISTICS ===")
print(f"Mean delay: {mean_delay:.2f} minutes")
print(f"Median delay: {median_delay:.2f} minutes")
print(f"Std deviation: {df['DEP_DELAY'].std():.2f} minutes")
print(f"Skewness: {df['DEP_DELAY'].skew():.2f}")
print(f"\nDelayed flights (≥15 min): {(df['DEP_DELAY'] >= 15).sum():,} ({(df['DEP_DELAY'] >= 15).sum()/len(df)*100:.1f}%)")
print(f"Early departures (<0 min): {(df['DEP_DELAY'] < 0).sum():,} ({(df['DEP_DELAY'] < 0).sum()/len(df)*100:.1f}%)")

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Distribution shape (normal, skewed, multimodal)
- Position of mean vs median (indicates skewness)
- Proportion of flights exceeding delay threshold
- Presence and extent of outliers

### Visualization 2: Delay Frequency by Time Block

**Purpose**: Identify which time periods of the day have the highest frequency of delays.

**Chart Type**: Bar chart with percentage annotations

In [ ]:
# Visualization 2: Delay Frequency by Time Block

# Calculate delayed flights by time block
time_block_delays = df[df['DEP_DELAY'] >= 15].groupby('Time_Block').size().reset_index(name='Delayed_Count')
time_block_total = df.groupby('Time_Block').size().reset_index(name='Total_Count')
time_block_summary = time_block_delays.merge(time_block_total, on='Time_Block')
time_block_summary['Delay_Rate'] = (time_block_summary['Delayed_Count'] / time_block_summary['Total_Count'] * 100)

# Order by logical time sequence
time_order = ['Morning', 'Afternoon', 'Evening', 'Night']
time_block_summary['Time_Block'] = pd.Categorical(time_block_summary['Time_Block'], categories=time_order, ordered=True)
time_block_summary = time_block_summary.sort_values('Time_Block')

# Create interactive bar chart with Plotly
fig = px.bar(time_block_summary, 
             x='Time_Block', 
             y='Delay_Rate',
             title='Flight Delay Rate by Time of Day<br><sub>Percentage of Flights Delayed ≥15 Minutes</sub>',
             labels={'Time_Block': 'Time Block', 'Delay_Rate': 'Delay Rate (%)'},
             text='Delay_Rate',
             color='Delay_Rate',
             color_continuous_scale='Reds')

# Update layout
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(showlegend=False, height=600, width=1000)

save_viz(fig, 'viz_02_time_block_delays')
fig.show()

print("\n=== DELAY RATE BY TIME BLOCK ===")
print(time_block_summary[['Time_Block', 'Delayed_Count', 'Total_Count', 'Delay_Rate']].to_string(index=False))

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Which time block has the highest delay rate
- Whether delays compound throughout the day
- Operational implications for scheduling

## Category 2: Bivariate Analysis - Trends

### Visualization 3: Airline Delay Performance Comparison

**Purpose**: Compare delay distributions across carriers to identify best and worst performers.

**Chart Type**: Violin plot (shows full distribution shape)

In [ ]:
# Visualization 3: Airline Performance Comparison

# Calculate median delay by carrier for sorting
carrier_medians = df.groupby('OP_CARRIER')['DEP_DELAY'].median().sort_values(ascending=False)

# Filter to carriers with sufficient sample size (>1000 flights)
carrier_counts = df['OP_CARRIER'].value_counts()
major_carriers = carrier_counts[carrier_counts > 1000].index
df_carriers = df[df['OP_CARRIER'].isin(major_carriers)]

# Limit delay range for better visualization
df_carriers_plot = df_carriers[df_carriers['DEP_DELAY'].between(-30, 120)]

# Create violin plot
fig, ax = plt.subplots(figsize=(14, 8))

sns.violinplot(data=df_carriers_plot, 
               y='OP_CARRIER', 
               x='DEP_DELAY',
               order=carrier_medians.index,
               palette='Set2',
               ax=ax)

# Add vertical line at delay threshold
ax.axvline(x=15, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Delay Threshold')
ax.axvline(x=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)

# Labels
ax.set_xlabel('Departure Delay (minutes)', fontsize=12)
ax.set_ylabel('Airline Carrier', fontsize=12)
ax.set_title('Airline Delay Performance Comparison\nDistribution of Departure Delays by Carrier (Sorted by Median)', 
             fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
save_viz(fig, 'viz_03_carrier_performance')
plt.show()

# Summary statistics
print("\n=== CARRIER PERFORMANCE SUMMARY ===")
carrier_stats = df_carriers.groupby('OP_CARRIER')['DEP_DELAY'].agg([
    ('Count', 'count'),
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std', 'std')
]).round(2)
carrier_stats = carrier_stats.sort_values('Median', ascending=False)
print(carrier_stats)

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Best performing carrier (lowest median delay)
- Worst performing carrier (highest median delay)
- Consistency (narrow vs wide distributions)
- Outlier patterns

### Visualization 4: Delay Severity Heatmap (Season × Haul Type)

**Purpose**: Show interaction between seasonal and distance factors on delay severity.

**Chart Type**: 2D Heatmap with diverging color scale

In [ ]:
# Visualization 4: Season × Haul Type Delay Heatmap

# Calculate mean arrival delay by season and haul type
heatmap_data = df.pivot_table(values='ARR_DELAY', 
                               index='Season', 
                               columns='Haul_Type', 
                               aggfunc='mean')

# Order columns logically
haul_order = ['Short-Haul', 'Medium-Haul', 'Long-Haul']
season_order = ['Winter', 'Spring', 'Summer', 'Fall']

heatmap_data = heatmap_data.reindex(index=season_order, columns=haul_order)

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(heatmap_data, 
            annot=True, 
            fmt='.1f', 
            cmap='RdYlGn_r',  # Red for delays, green for early
            center=0,  # Center diverging colormap at 0
            cbar_kws={'label': 'Mean Arrival Delay (minutes)'},
            linewidths=0.5,
            ax=ax)

ax.set_title('Delay Severity by Season and Flight Distance\nMean Arrival Delay (minutes)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Flight Distance Category', fontsize=12)
ax.set_ylabel('Season', fontsize=12)

plt.tight_layout()
save_viz(fig, 'viz_04_season_haul_heatmap')
plt.show()

print("\n=== MEAN ARRIVAL DELAY BY SEASON AND HAUL TYPE ===")
print(heatmap_data)

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Which season/distance combinations have worst delays
- Whether long-haul flights make up time in air
- Seasonal patterns across different flight types

### Visualization 5: Daily Volume vs Average Delay Trend

**Purpose**: Test if delays increase during periods of high flight volume.

**Chart Type**: Dual-axis time series with Plotly for interactivity

In [ ]:
# Visualization 5: Daily Volume vs Delay Trend

# Aggregate by date
if date_col in df.columns:
    daily_stats = df.groupby(date_col).agg({
        'DEP_DELAY': 'mean',
        'ORIGIN': 'count'  # Count flights
    }).reset_index()
    daily_stats.columns = ['Date', 'Mean_Delay', 'Flight_Count']
    
    # Create dual-axis plot with Plotly
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Add mean delay line
    fig.add_trace(
        go.Scatter(x=daily_stats['Date'], y=daily_stats['Mean_Delay'], 
                   name='Mean Delay', mode='lines', line=dict(color='red', width=2)),
        secondary_y=False
    )
    
    # Add flight count line
    fig.add_trace(
        go.Scatter(x=daily_stats['Date'], y=daily_stats['Flight_Count'], 
                   name='Daily Flights', mode='lines', line=dict(color='blue', width=2)),
        secondary_y=True
    )
    
    # Update layout
    fig.update_xaxes(title_text="Date")
    fig.update_yaxes(title_text="Mean Departure Delay (minutes)", secondary_y=False)
    fig.update_yaxes(title_text="Number of Flights", secondary_y=True)
    fig.update_layout(
        title="Daily Flight Volume vs. Average Delay Trend<br><sub>Examining Correlation Between Operational Load and Delays</sub>",
        hovermode='x unified',
        height=600,
        width=1200
    )
    
    save_viz(fig, 'viz_05_volume_delay_trend')
    fig.show()
    
    # Calculate correlation
    correlation = daily_stats['Mean_Delay'].corr(daily_stats['Flight_Count'])
    print(f"\nCorrelation between daily volume and mean delay: {correlation:.3f}")
else:
    print("⚠️  Date column not available for time series analysis")

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Correlation strength between volume and delays
- Temporal patterns (weekday vs weekend, holidays)
- Capacity threshold effects

## Category 3: Geospatial/Route Analysis

### Visualization 6: Top 10 Bottleneck Routes

**Purpose**: Identify consistently problematic routes with highest delays.

**Chart Type**: Horizontal bar chart with flight count annotations

In [ ]:
# Visualization 6: Top Bottleneck Routes

# Calculate mean delay and flight count by route
route_stats = df.groupby('Route_ID').agg({
    'ARR_DELAY': 'mean',
    'ORIGIN': 'count'
}).reset_index()
route_stats.columns = ['Route_ID', 'Mean_Delay', 'Flight_Count']

# Filter routes with at least 100 flights for statistical significance
route_stats = route_stats[route_stats['Flight_Count'] >= 100]

# Get top 10 worst routes by delay
top_10_worst = route_stats.nlargest(10, 'Mean_Delay')

# Create horizontal bar chart with Plotly
fig = px.bar(top_10_worst.sort_values('Mean_Delay'), 
             y='Route_ID', 
             x='Mean_Delay',
             orientation='h',
             title='Top 10 Most Delayed Routes<br><sub>Routes with ≥100 Flights, Ranked by Mean Arrival Delay</sub>',
             labels={'Route_ID': 'Route', 'Mean_Delay': 'Mean Arrival Delay (minutes)'},
             text='Mean_Delay',
             color='Mean_Delay',
             color_continuous_scale='Reds')

# Add flight count as custom text
fig.update_traces(texttemplate='%{text:.1f} min<br>(%{customdata[0]:,} flights)', 
                  textposition='outside',
                  customdata=top_10_worst[['Flight_Count']].values)

fig.update_layout(showlegend=False, height=600, width=1000)

save_viz(fig, 'viz_06_bottleneck_routes')
fig.show()

print("\n=== TOP 10 BOTTLENECK ROUTES ===")
print(top_10_worst[['Route_ID', 'Mean_Delay', 'Flight_Count']].to_string(index=False))

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Geographic patterns (e.g., weather-prone regions)
- Hub congestion effects
- Specific operational challenges on these routes

### Visualization 7: Hub vs Non-Hub Delay Comparison

**Purpose**: Test if major hub airports have worse delay characteristics.

**Chart Type**: Split violin plot showing full distributions

In [ ]:
# Visualization 7: Hub vs Non-Hub Comparison

# Prepare data (limit range for visualization)
df_hub_plot = df[df['DEP_DELAY'].between(-30, 120)].copy()
df_hub_plot['Hub_Status'] = df_hub_plot['Is_Hub'].map({True: 'Hub Airport', False: 'Non-Hub Airport'})

# Create violin plot
fig, ax = plt.subplots(figsize=(12, 7))

sns.violinplot(data=df_hub_plot, 
               x='Hub_Status', 
               y='DEP_DELAY',
               palette=['#e74c3c', '#3498db'],
               inner='quartile',
               ax=ax)

# Add mean markers
means = df_hub_plot.groupby('Hub_Status')['DEP_DELAY'].mean()
positions = range(len(means))
ax.scatter(positions, means, color='yellow', s=200, zorder=3, 
           edgecolors='black', linewidths=2, label='Mean')

# Add delay threshold line
ax.axhline(y=15, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Delay Threshold (15 min)')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)

# Labels
ax.set_xlabel('Airport Type', fontsize=12)
ax.set_ylabel('Departure Delay (minutes)', fontsize=12)
ax.set_title('Hub vs Non-Hub Airport Delay Comparison\nDistribution of Departure Delays by Airport Type', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_viz(fig, 'viz_07_hub_comparison')
plt.show()

# Statistical summary
print("\n=== HUB VS NON-HUB STATISTICS ===")
hub_stats = df.groupby('Is_Hub')['DEP_DELAY'].agg([
    ('Count', 'count'),
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std', 'std'),
    ('Delay_Rate_%', lambda x: (x >= 15).sum() / len(x) * 100)
]).round(2)
hub_stats.index = ['Non-Hub', 'Hub']
print(hub_stats)

# Simple t-test for statistical significance
from scipy import stats
hub_delays = df[df['Is_Hub'] == True]['DEP_DELAY'].dropna()
nonhub_delays = df[df['Is_Hub'] == False]['DEP_DELAY'].dropna()
t_stat, p_value = stats.ttest_ind(hub_delays, nonhub_delays)
print(f"\nt-test: t={t_stat:.3f}, p={p_value:.4f}")
print(f"Statistically significant difference: {'Yes' if p_value < 0.05 else 'No'}")

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Whether hub airports have significantly higher delays
- Distribution shape differences
- Operational implications for hub-and-spoke network model

## Category 4: Correlation & Advanced Analysis

### Visualization 8: Airport Congestion Impact

**Purpose**: Quantify relationship between airport daily volume and delays.

**Chart Type**: Scatter plot with regression line

In [ ]:
# Visualization 8: Airport Congestion Impact

# Aggregate to airport-day level
airport_day_stats = df.groupby(['ORIGIN', date_col]).agg({
    'DEP_DELAY': 'mean',
    'Daily_Airport_Departures': 'first',
    'Is_Hub': 'first'
}).reset_index()

airport_day_stats['Hub_Status'] = airport_day_stats['Is_Hub'].map({True: 'Hub', False: 'Non-Hub'})

# Sample for performance (use 10% of data points)
sample_size = min(10000, len(airport_day_stats))
plot_data = airport_day_stats.sample(n=sample_size, random_state=42)

# Create scatter plot with regression
fig, ax = plt.subplots(figsize=(12, 7))

# Plot by hub status
for hub_status, color in [('Hub', '#e74c3c'), ('Non-Hub', '#3498db')]:
    data_subset = plot_data[plot_data['Hub_Status'] == hub_status]
    ax.scatter(data_subset['Daily_Airport_Departures'], 
               data_subset['DEP_DELAY'],
               alpha=0.5, 
               s=30, 
               color=color,
               label=hub_status)

# Add regression line for all data
sns.regplot(data=plot_data, 
            x='Daily_Airport_Departures', 
            y='DEP_DELAY',
            scatter=False,
            color='black',
            line_kws={'linewidth': 2, 'label': 'Regression Line'},
            ax=ax)

# Calculate and display correlation
correlation = plot_data['Daily_Airport_Departures'].corr(plot_data['DEP_DELAY'])

# Labels
ax.set_xlabel('Daily Airport Departures', fontsize=12)
ax.set_ylabel('Mean Departure Delay (minutes)', fontsize=12)
ax.set_title(f'Airport Congestion Impact on Delays\nCorrelation: {correlation:.3f}', 
             fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=1, alpha=0.5)

plt.tight_layout()
save_viz(fig, 'viz_08_congestion_impact')
plt.show()

print(f"\nCorrelation coefficient: {correlation:.4f}")
print(f"Sample size: {len(plot_data):,} airport-day observations")

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Strength of volume-delay relationship
- Whether hubs show different patterns
- Evidence of capacity thresholds

### Visualization 9: Carrier Load vs Performance

**Purpose**: Test if airline daily utilization affects delay performance.

**Chart Type**: Interactive scatter plot with bubble size

In [ ]:
# Visualization 9: Carrier Load vs Performance

# Aggregate to carrier-day level
carrier_day_stats = df.groupby(['OP_CARRIER', date_col]).agg({
    'DEP_DELAY': 'mean',
    'Daily_Carrier_Load': 'first',
    'ORIGIN': lambda x: (x >= 15).sum()  # Count of delayed flights (this is approximate)
}).reset_index()
carrier_day_stats.columns = ['Carrier', 'Date', 'Mean_Delay', 'Daily_Load', 'Delayed_Flights']

# Sample for performance
sample_size = min(5000, len(carrier_day_stats))
plot_data_carrier = carrier_day_stats.sample(n=sample_size, random_state=42)

# Create interactive scatter with Plotly
fig = px.scatter(plot_data_carrier, 
                 x='Daily_Load', 
                 y='Mean_Delay',
                 size='Delayed_Flights',
                 color='Carrier',
                 hover_data=['Date'],
                 title='Carrier Daily Load vs. Delay Performance<br><sub>Bubble Size = Number of Delayed Flights</sub>',
                 labels={'Daily_Load': 'Daily Carrier Load (# Flights)',
                        'Mean_Delay': 'Mean Departure Delay (minutes)'})

# Add trend line
fig.add_trace(go.Scatter(
    x=plot_data_carrier['Daily_Load'],
    y=plot_data_carrier['Daily_Load'].apply(lambda x: np.poly1d(np.polyfit(plot_data_carrier['Daily_Load'], 
                                                                             plot_data_carrier['Mean_Delay'], 1))(x)),
    mode='lines',
    name='Trend',
    line=dict(color='black', width=2, dash='dash')
))

fig.update_layout(height=600, width=1200)

save_viz(fig, 'viz_09_carrier_load')
fig.show()

# Correlation
correlation_carrier = plot_data_carrier['Daily_Load'].corr(plot_data_carrier['Mean_Delay'])
print(f"\nCorrelation between carrier load and delay: {correlation_carrier:.4f}")

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Whether overextended carriers have worse performance
- Carrier-specific patterns
- Optimal vs excessive utilization

### Visualization 10: Multi-Dimensional Route Analysis (Bonus)

**Purpose**: Show complex interaction between distance, time, and season.

**Chart Type**: Faceted scatter plots

In [ ]:
# Visualization 10: Multi-Dimensional Analysis

# Sample data for performance
sample_data = df.sample(n=min(20000, len(df)), random_state=42)

# Create faceted scatter plot
g = sns.FacetGrid(sample_data, 
                  col='Season', 
                  row='Time_Block',
                  height=3, 
                  aspect=1.2,
                  col_order=['Winter', 'Spring', 'Summer', 'Fall'],
                  row_order=['Morning', 'Afternoon', 'Evening', 'Night'])

g.map(sns.scatterplot, 'DISTANCE', 'ARR_DELAY', alpha=0.3, s=10)

# Add regression lines
g.map(sns.regplot, 'DISTANCE', 'ARR_DELAY', scatter=False, color='red', line_kws={'linewidth': 2})

# Add delay threshold line
for ax in g.axes.flat:
    ax.axhline(y=15, color='orange', linestyle='--', linewidth=1, alpha=0.5)
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

g.set_axis_labels('Flight Distance (miles)', 'Arrival Delay (minutes)')
g.fig.suptitle('Multi-Dimensional Delay Analysis: Distance × Season × Time Block', 
               fontsize=16, fontweight='bold', y=1.01)

plt.tight_layout()
save_viz(g.fig, 'viz_10_multidimensional')
plt.show()

print("✓ Multi-dimensional visualization created")

**Interpretation**:

*[To be filled after execution]*

Expected insights:
- Complex interaction patterns
- Context-dependent relationships
- Most problematic combinations of factors

---
<a id='part4'></a>
# PART 4: INSIGHTS & FINDINGS SUMMARY

## Section 4.1: Key Insights

*[To be completed after running all visualizations]*

### Top 3 Most Important Findings:

1. **[Finding 1]**
   - Evidence: [Reference to specific visualization]
   - Implication: [Operational or analytical significance]

2. **[Finding 2]**
   - Evidence: [Reference to specific visualization]
   - Implication: [Operational or analytical significance]

3. **[Finding 3]**
   - Evidence: [Reference to specific visualization]
   - Implication: [Operational or analytical significance]

### Patterns Identified:

**Temporal Patterns**:
- [Description of time-based patterns]

**Geographic Patterns**:
- [Description of location-based patterns]

**Operational Patterns**:
- [Description of volume/capacity patterns]

### Anomalies or Unexpected Results:

- [List any surprising findings]
- [Potential explanations]

### Data Limitations Encountered:

- [Note any data quality issues]
- [Missing information that would enhance analysis]

## Section 4.2: Analytical Comparisons

### Visualization Approach Decisions:

**1. Distribution Visualization**:
- Tried: Histogram, KDE, Box plot
- Selected: Histogram with overlaid KDE
- Rationale: Shows both frequency and smooth distribution shape
- Trade-off: More complex than simple histogram, but more informative

**2. Comparison Visualization**:
- Tried: Box plot, Violin plot, Strip plot
- Selected: Violin plot
- Rationale: Shows full distribution shape, not just quartiles
- Trade-off: Requires more space, but reveals multimodal patterns

**3. Relationship Visualization**:
- Tried: Scatter plot, Hexbin, 2D histogram
- Selected: Scatter with regression line
- Rationale: Shows individual points and overall trend
- Trade-off: Can be cluttered with large samples (addressed via sampling)

### Library Selection:

**Matplotlib/Seaborn**:
- Used for: Static, publication-quality visualizations
- Advantages: Fine-grained control, excellent for reports
- Limitations: Less interactive

**Plotly**:
- Used for: Interactive exploration, complex multi-variate plots
- Advantages: Hover tooltips, zoom, interactive legends
- Limitations: Larger file sizes, requires JavaScript for web viewing

## Section 4.3: Recommendations for Further Analysis

### Additional Features to Engineer:

1. **Weather Integration**:
   - Incorporate historical weather data (precipitation, visibility, wind)
   - Create weather severity index
   - Test correlation with delays

2. **Aircraft-Level Features**:
   - Aircraft age
   - Aircraft type (narrow-body vs wide-body)
   - Turnaround time between flights

3. **Network Features**:
   - Upstream delay propagation (previous flight delay)
   - Network centrality of airports
   - Connection banking patterns

### External Data Sources:

- **NOAA Weather Data**: Historical weather observations
- **FAA Airport Data**: Runway configurations, capacity limits
- **Holiday Calendars**: Federal holidays, school breaks
- **Economic Indicators**: Fuel prices, GDP (for demand modeling)

### Statistical Tests to Validate Patterns:

1. **ANOVA**: Test significance of differences across time blocks/seasons
2. **Chi-square**: Test independence of categorical variables
3. **Granger Causality**: Test temporal causation in delay propagation
4. **Regression Analysis**: Quantify effect sizes with confidence intervals

### Predictive Modeling Opportunities:

1. **Classification Models**:
   - Predict probability of delay (binary)
   - Multi-class: On-time, Minor delay, Major delay, Cancelled

2. **Regression Models**:
   - Predict delay duration (continuous)
   - Quantile regression for worst-case scenarios

3. **Advanced Techniques**:
   - Time series forecasting (ARIMA, Prophet)
   - Ensemble methods (Random Forest, Gradient Boosting)
   - Deep learning (LSTM for sequential patterns)

### Operational Applications:

- **Schedule Optimization**: Identify high-risk time slots for buffer time allocation
- **Resource Allocation**: Prioritize ground crew during peak delay periods
- **Customer Communication**: Proactive delay notifications for high-risk flights
- **Network Design**: Evaluate hub-and-spoke vs point-to-point trade-offs

## Final Summary

### Phase 2 Completion Checklist:

✅ **Data Loading**: Successfully loaded Phase 1 output  
✅ **Feature Engineering**: Created 7 new features  
✅ **Visualizations**: Completed 10 complex visualizations  
✅ **Insights**: Documented key findings and patterns  
✅ **Export**: Saved all visualizations to reports/  
✅ **Documentation**: Comprehensive markdown explanations  

### Deliverables:

1. Consolidated EDA notebook with all code and outputs
2. 10 publication-quality visualizations
3. Feature-engineered dataset with 7 new columns
4. Comprehensive insights and recommendations

### Next Phase:

**Phase 3: Advanced Analytics & Modeling**
- Predictive delay models
- Statistical validation
- Interactive dashboard
- Final report compilation

---

**✓ Phase 2 Complete: Feature Engineering & Exploratory Data Analysis**